# Potential flow

AE2130-I, Delft University of Technology.

In [125]:
# page:hide
%pip install -q plotly ipywidgets

import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from plotly.subplots import make_subplots

In [126]:
# page:hide
from scipy.interpolate import RectBivariateSpline
from scipy.interpolate import RegularGridInterpolator

def rk4_step(f, xn, h):
    """One classical RK4 step for dx/ds = f(x)."""
    k1 = f(xn)
    k2 = f(xn + h/2*k1)
    k3 = f(xn + h/2*k2)
    k4 = f(xn + h*k3)
    return xn + h/6*(k1 + 2*k2 + 2*k3 + k4)

def _trace(f, x0, ds, n_max, inside):
    """March from x0 with RK4 steps of size ds (negative = backwards)."""
    pts = [np.asarray(x0, float)]
    for _ in range(n_max):
        xn  = pts[-1]
        xn1 = rk4_step(f, xn, ds)
        if not inside(xn1) or np.allclose(xn1, xn):
            break
        pts.append(xn1)
    return np.array(pts)

def streamline(x0, uv, xy, ds=0.01, n_fwd=300, n_back=300):
    (u, v), (x, y) = uv, xy
    U = RectBivariateSpline(x[::2], y[::2], u[::2, ::2], kx=3, ky=3)   # cubic interpolation
    V = RectBivariateSpline(x[::2], y[::2], v[::2, ::2], kx=3, ky=3)

    def f(xn):                                     # unit velocity direction
        vel = np.array([U(*xn)[0, 0], V(*xn)[0, 0]])
        sp = np.linalg.norm(vel)
        return vel / sp if sp > 1e-12 else np.zeros(2)

    def inside(xn):
        return x[0] <= xn[0] <= x[-1] and y[0] <= xn[1] <= y[-1]

    fwd  = _trace(f, x0,  ds, n_fwd,  inside)
    back = _trace(f, x0, -ds, n_back, inside)
    return np.vstack([back[::-1], fwd[1:]]).T     # back … x0 … forward


This notebook serves to showcase the potential flow module taught in AE2130-I. It is an interactive notebook, so you can modify the code yourself!

## Uniform Flow
<!-- page:closed -->

Let's start with the simplest elementary flow. Consider a flow at an angle $\alpha$ with components

$$
\vec{u} = \begin{pmatrix} u \\ v \end{pmatrix}
        = \begin{pmatrix} V_{\infty}\cos\alpha \\ V_{\infty}\sin\alpha \end{pmatrix}.
$$

This is a **uniform flow**: the velocity is the same everywhere in the domain. It is both divergence free, $\vec{\nabla}\cdot\vec{u} = 0$, and irrotational, $\vec{\nabla}\times\vec{u} = 0$. The velocity potential $\phi$ must satisfy

$$
\frac{\partial \phi}{\partial x} = u = V_{\infty}\cos\alpha,
\qquad
\frac{\partial \phi}{\partial y} = v = V_{\infty}\sin\alpha.
$$

Integrating each equation gives

$$
\begin{array}{r@{\;}c@{\;}c@{\;}c@{\;}l}
\phi(x,y) = & V_{\infty}\, x\cos\alpha & + & C(y)         & +\; C, \\[6pt]
            & \Big\downarrow           &   & \Big\uparrow & \\[6pt]
\phi(x,y) = & C(x)                     & + & V_{\infty}\, y\sin\alpha & +\; C.
\end{array}
$$

The integration constant of the first line can still depend on $y$, and that of the second on $x$. Both integrals describe the same $\phi$, so they must match term by term: $C(x) = V_{\infty}\, x\cos\alpha$ and $C(y) = V_{\infty}\, y\sin\alpha$. Hence

$$
\phi(x,y) = V_{\infty}\left(x\cos\alpha + y\sin\alpha\right) + C.
$$

We can do the same for the stream function, using $u = \partial\psi/\partial y$ and $v = -\partial\psi/\partial x$. Dropping the constants, the uniform flow is then described by

$$
\boxed{
\begin{aligned}
\phi(x,y) &= V_{\infty} \Big( x\cos\alpha + y\sin\alpha \Big) \\[6pt]
\psi(x,y) &= V_{\infty} \Big( y\cos\alpha - x\sin\alpha \Big)
\end{aligned}
}
$$

## Source Flow
<!-- page:closed -->

## Doublet Flow
<!-- page:closed -->

## Vortex Flow
<!-- page:closed -->

# Combinations of flows

We now make an effort to displaying a series of combinations of potential flows

In [127]:
# page:fold
x = np.linspace(-1,1,200)
y = np.linspace(-1,1,200)

X, Y     = np.meshgrid(x,y, indexing='ij')
R, THETA = np.hypot(X,Y), np.arctan2(Y,X)

In [128]:
# page:fold
def uniform(Vinf,angle, x=X,y=Y):
    Vx, Vy = Vinf*np.cos(angle), Vinf*np.sin(angle)

    phi = Vx*x + Vy*y
    psi = Vx*y - Vy*x

    return phi, psi

def source(Lambda, r=R,theta=THETA):
    strength = Lambda/(2*np.pi)

    phi = strength*np.log(r)
    psi = strength*theta

    return phi, psi

def doublet(kappa, r=R,theta=THETA):
    strength = kappa/(2*np.pi)

    phi =  strength*np.cos(theta)/r
    psi = -strength*np.sin(theta)/r

    return phi, psi

def vortex(Gamma, r=R,theta=THETA):
    strength = Gamma/(2*np.pi)

    phi = -strength*theta
    psi =  strength*np.log(r)

    return phi, psi

In [129]:
# page:fold
def ddx(f,x=x): return np.gradient(f, x, axis=0)
def ddy(f,x=y): return np.gradient(f, y, axis=1)

## Semi-infinite Body
<!-- page:closed -->

_To be written._

In [130]:
# page:hide
# page:rerun=1

# page:slider (0.5, 5.0, 0.1)
Vinf = 1

# page:slider (-30, 30, 1)
angle = 0

# page:slider (1, 5, 0.1)
Lambda = 1

In [131]:
phi_1, psi_1 = uniform(Vinf,np.deg2rad(angle))
phi_2, psi_2 = source(Lambda)

phi_t = phi_1 + phi_2
psi_t = psi_1 + psi_2

u = ddx(phi_t)
v = ddy(phi_t)

V  = np.hypot(u, v)
Cp = 1 - (V/Vinf)**2

# page:hide-from
fig = go.Figure()
def draw_streamline(x0, width=1, **opts):
    xs, ys = streamline(x0, (u, v), (x, y), **opts)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color="white", width=width), hoverinfo="skip")

## -------------- ##
## Pressure Field ##
## -------------- ##
dark_rdbu_r = [[0.0, "#4a86c5"],    # light blue  (Cp = -1, suction)
               [0.5, "#1a1a1a"],   # white       (Cp = 0)
               [1.0, "#d9605f"]]   # light red   (Cp = 1, stagnation)
fig.add_contour(x=x, y=y, z=Cp.T,
                contours=dict(start=-1, end=1, size=0.125, coloring="fill"),
                line_width=0, colorscale=dark_rdbu_r, zmin=-1, zmax=1, zmid=0,
                colorbar=dict(title="Cp"), hoverinfo="skip")

## ----------------------- ##
## free-stream streamlines ##
## ----------------------- ##
a  = np.deg2rad(angle)
c  = -0.9*np.array([np.cos(a), np.sin(a)])       # centre of the line, upstream
e  =  0.9*np.array([-np.sin(a), np.cos(a)])      # half-length along the perpendicular
seeds = np.linspace(c - e, c + e, 20)            # (21, 2) points from one end to the other

for x0 in seeds:
    draw_streamline(x0, width=1)

## ------------------ ##
## source streamlines ##
## ------------------ ##
R, n = 0.05, 8                                   # circle radius, number of seeds
ang = np.linspace(0, 2*np.pi, n, endpoint=False)
ring = np.array([0.0, 0.0]) + R*np.column_stack([np.cos(ang), np.sin(ang)])

for x0 in ring:
    draw_streamline(x0, width=1)
fig.add_scatter(x=[0], y=[0], mode="markers", marker=dict(size=10, color="red"), name="source")

## ---------------- ##
## stagnation point ##
## ---------------- ##
a = np.deg2rad(angle)
d = Lambda / (2*np.pi*Vinf)
t      = np.array([np.cos(a), np.sin(a)])     # freestream direction
nrm    = np.array([-np.sin(a), np.cos(a)])    # perpendicular to it
x_stag = -d * t                               # stagnation point
eps = 0.01

seeds = [(x_stag - eps*t,   dict(n_fwd=0)),   # incoming arm: trace backwards upstream
         (x_stag + eps*nrm, dict(n_back=0)),  # upper branch: trace forwards
         (x_stag - eps*nrm, dict(n_back=0))]  # lower branch: trace forwards

for x0, opts in seeds:
    draw_streamline(x0, width=3, **opts)

fig.add_scatter(x=[x_stag[0]], y=[x_stag[1]], mode="markers",
                marker=dict(symbol="star", size=16, color="black",
                            line=dict(color="white", width=1.5)),
                showlegend=False)

fig.update_layout(title="Uniform flow + source", width=700, height=600, showlegend=False,
                  xaxis=dict(range=[-1, 1], constrain="domain"), yaxis=dict(range=[-1, 1], scaleanchor="x"))
fig.update_layout(template="plotly_dark",
                  paper_bgcolor="rgba(0,0,0,0)",   # transparent: shows the page's black
                  plot_bgcolor="rgba(0,0,0,0)",
                  font_color="#e0e0e0")
fig.update_xaxes(fixedrange=True)          # no zooming or panning on any axis
fig.update_yaxes(fixedrange=True)
fig.update_layout(dragmode=False)          # dragging doesn't box-select either
fig.show(config={"displayModeBar": False, "scrollZoom": False})
# page:hide-to

## Rankine Oval
<!-- page:closed -->

_To be written._

In [132]:
# page:hide
# page:rerun=1

# page:slider (0.5, 5.0, 0.1)
Vinf = 1

# page:slider (0.05, 1.0, 0.05)
sep  = 1.0

# page:slider (1, 5, 0.1)
Lambda = 1

In [133]:
R2, THETA2 = np.hypot(X+sep/2,Y), np.arctan2(Y,X+sep/2)
R3, THETA3 = np.hypot(X-sep/2,Y), np.arctan2(Y,X-sep/2)

phi_1, psi_1 = uniform(Vinf,0)
phi_2, psi_2 = source( Lambda, r=R2, theta=THETA2)
phi_3, psi_3 = source(-Lambda, r=R3, theta=THETA3)

phi_t = phi_1 + phi_2 + phi_3
psi_t = psi_1 + psi_2 + psi_3

u = ddx(phi_t)
v = ddy(phi_t)

V  = np.hypot(u, v)
Cp = 1 - (V/Vinf)**2

# page:hide-from
fig = go.Figure()
def draw_streamline(x0, width=1, **opts):
    xs, ys = streamline(x0, (u, v), (x, y), **opts)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color="white", width=width), hoverinfo="skip")

## -------------- ##
## Pressure Field ##
## -------------- ##
dark_rdbu_r = [[0.0, "#4a86c5"],    # blue (Cp = -1, suction)
               [0.5, "#1a1a1a"],    # dark (Cp = 0)
               [1.0, "#d9605f"]]    # red  (Cp = 1, stagnation)
fig.add_contour(x=x, y=y, z=Cp.T,
                contours=dict(start=-1, end=1, size=0.125, coloring="fill"),
                line_width=0, colorscale=dark_rdbu_r, zmin=-1, zmax=1,
                colorbar=dict(title="Cp"), hoverinfo="skip")

## ---------------------------------- ##
## free-stream + internal streamlines ##
## ---------------------------------- ##
for y0 in np.linspace(-1, 1, 21)[1:-1]:    # seeds on the line x = 0, traced both ways
    draw_streamline(np.array([0.0, y0]))

## ----------------- ##
## stagnation points ##
## ----------------- ##
d = np.sqrt((sep/2)**2 + Lambda*(sep/2)/(np.pi*Vinf))   # distance from the centre
front, rear = np.array([-d, 0.0]), np.array([d, 0.0])
eps = 0.01

seeds = [(front + np.array([-eps, 0]), dict(n_fwd=0)),   # incoming arm: trace backwards upstream
         (front + np.array([0,  eps]), dict(n_back=0)),  # upper half of the oval
         (front + np.array([0, -eps]), dict(n_back=0)),  # lower half of the oval
         (rear  + np.array([ eps, 0]), dict(n_back=0))]  # outgoing arm: trace forwards downstream

for x0, opts in seeds:
    draw_streamline(x0, width=3, **opts)

fig.add_scatter(x=[front[0], rear[0]], y=[0, 0], mode="markers",
                marker=dict(symbol="star", size=16, color="black",
                            line=dict(color="white", width=1.5)))
fig.add_scatter(x=[-sep/2, sep/2], y=[0, 0], mode="markers",
                marker=dict(size=10, color=["red", "blue"]))   # source red, sink blue

fig.update_layout(title="Rankine oval: uniform flow + source + sink", width=700, height=600,
                  showlegend=False,
                  xaxis=dict(range=[-1, 1], constrain="domain"),
                  yaxis=dict(range=[-1, 1], scaleanchor="x"))
fig.update_layout(template="plotly_dark",
                  paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                  font_color="#e0e0e0")
fig.update_xaxes(fixedrange=True)          # no zooming or panning on any axis
fig.update_yaxes(fixedrange=True)
fig.update_layout(dragmode=False)          # dragging doesn't box-select either
fig.show(config={"displayModeBar": False, "scrollZoom": False})
# page:hide-to

## Non-lifting flow over a cylinder
<!-- page:closed -->

You can see as you reduce the separation $s$ while increasing $\Lambda$ such that $s\Lambda = \mathrm{const}$, that the Rankine oval becomes more-and-more like a circle!

In [134]:
# page:hide
# page:rerun=1

# page:slider (0.5, 5.0, 0.1)
Vinf = 1

# page:slider (0.1, 0.6, 0.05)
R = 0.4

In [124]:
kappa = 2*np.pi*Vinf*R**2                            # doublet strength that gives a cylinder of radius R

phi_1, psi_1 = uniform(Vinf, 0)
phi_2, psi_2 = doublet(kappa)

phi_t = phi_1 + phi_2
psi_t = psi_1 + psi_2

u = ddx(phi_t)
v = ddy(phi_t)

V  = np.hypot(u, v)
Cp = 1 - (V/Vinf)**2

# page:hide-from
Cp_plot = np.where(np.hypot(X, Y) < R, np.nan, Cp)   # blank out the inside of the cylinder

fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.22,
                    subplot_titles=("Flow field", "Surface pressure"))

def draw_streamline(x0, width=1, **opts):
    xs, ys = streamline(x0, (u, v), (x, y), **opts)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color="white", width=width),
                    hoverinfo="skip", row=1, col=1)

## -------------- ##
## Pressure Field ##
## -------------- ##
dark_rdbu_r = [[0.0, "#4a86c5"], [0.5, "#1a1a1a"], [1.0, "#d9605f"]]
fig.add_contour(x=x, y=y, z=Cp_plot.T,
                contours=dict(start=-3, end=1, size=0.25, coloring="fill"),
                line_width=0, colorscale=dark_rdbu_r, zmin=-3, zmax=1, zmid=0,
                colorbar=dict(title="Cp", x=0.42, len=1, thickness=14), hoverinfo="skip", row=1, col=1)

## ----------------------- ##
## free-stream streamlines ##
## ----------------------- ##
for y0 in np.linspace(-0.95, 0.95, 20):
    if abs(y0) > 1e-3:                     # skip y = 0, it runs straight into the stagnation point
        draw_streamline(np.array([-0.99, y0]))

## --------------------------------------- ##
## cylinder surface (dividing streamline)  ##
## --------------------------------------- ##
t = np.linspace(0, 2*np.pi, 200)
fig.add_scatter(x=R*np.cos(t), y=R*np.sin(t), mode="lines", line=dict(color="white", width=3), hoverinfo="skip", row=1, col=1)
fig.add_scatter(x=[-1, -R], y=[0, 0], mode="lines", line=dict(color="white", width=3), hoverinfo="skip", row=1, col=1)
fig.add_scatter(x=[R, 1],  y=[0, 0], mode="lines", line=dict(color="white", width=3), hoverinfo="skip", row=1, col=1)

## ----------------- ##
## stagnation points ##
## ----------------- ##
fig.add_scatter(x=[-R, R], y=[0, 0], mode="markers",
                marker=dict(symbol="star", size=16, color="black",
                            line=dict(color="white", width=1.5)), row=1, col=1)

## ------------ ##
## theta marker ##
## ------------ ##
th0 = np.deg2rad(50)                                  # angle shown in the sketch
ref = dict(mode="lines", hoverinfo="skip", showlegend=False, row=1, col=1)

fig.add_scatter(x=[0, R], y=[0, 0], line=dict(color="white", width=1, dash="dot"), **ref)                  # θ = 0 reference
fig.add_scatter(x=[0, R*np.cos(th0)], y=[0, R*np.sin(th0)], line=dict(color="white", width=1.5), **ref)   # radius to the surface
arc = np.linspace(0, th0, 30)
fig.add_scatter(x=0.4*R*np.cos(arc), y=0.4*R*np.sin(arc), line=dict(color="white", width=1.5), **ref)     # the arc

fig.add_annotation(x=0.55*R*np.cos(th0/2), y=0.55*R*np.sin(th0/2), text="θ", showarrow=False,
                   font=dict(color="white", size=16), row=1, col=1)

## --------------------- ##
## Cp across the surface ##
## --------------------- ##
theta = np.linspace(0, 2*np.pi, 361)
Cp_at = RegularGridInterpolator((x, y), Cp)            # Cp is indexed [x, y]
Cp_num = Cp_at(np.column_stack([R*np.cos(theta), R*np.sin(theta)]))
Cp_exact = 1 - 4*np.sin(theta)**2

fig.add_scatter(x=np.rad2deg(theta), y=Cp_exact, mode="lines", line=dict(color="white", width=2),
                name="exact", row=1, col=2)
fig.add_scatter(x=np.rad2deg(theta[::15]), y=Cp_num[::15], mode="markers",
                marker=dict(color="#d9605f", size=7), name="from field", row=1, col=2)

# sized for the page's column (about 730 px wide): the square flow field, the
# colour bar and the Cp graph then all have the same height
fig.update_layout(title="Non-lifting cylinder: uniform flow + doublet", width=730, height=400,
                  margin=dict(l=40, r=10, t=80, b=50),
                  showlegend=False,
                  xaxis=dict(range=[-1, 1], constrain="domain"),
                  yaxis=dict(range=[-1, 1], scaleanchor="x"))
fig.update_layout(template="plotly_dark",
                  paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                  font_color="#e0e0e0")
fig.update_xaxes(fixedrange=True)          # no zooming or panning on any axis
fig.update_yaxes(fixedrange=True)
fig.update_layout(dragmode=False)          # dragging doesn't box-select either
fig.update_yaxes(range=[-1, 1], scaleanchor="x", constrain="domain", row=1, col=1)
fig.update_xaxes(title="θ [deg]", tickvals=[0, 90, 180, 270, 360], row=1, col=2)
fig.update_yaxes(title="Cp", row=1, col=2)
fig.show(config={"displayModeBar": False, "scrollZoom": False})
# page:hide-to

## Lifting flow over a cylinder
<!-- page:closed -->

_To be written._

<!-- page:closed -->
## Vortex panel method on a NACA airfoil

_To be written._